# Arcadia images: canonical Corva and dream paintings (SDXL on a Kaggle T4)

Generates the 14 images in `render/image_prompts.json` with SDXL base 1.0 (float16, with the fp16-fix VAE):
8 candidate "canonical Corva" paintings after `assets/reference/corva_reference.jpg`, and 6 surreal paintings of
the best dreams from `out/priming/run2.md`. Every image is saved as `<id>.png` with `<id>.json` (prompts, seed,
model, source dream).

Settings: **GPU T4**, **Internet on** (downloads the repo and ~7 GB of SDXL weights).

## 1. Clone the repo

In [ ]:
import os
if not os.path.exists("/kaggle/working/Arcadia"):
    !git clone --depth 1 https://github.com/DanielGregory/Arcadia /kaggle/working/Arcadia
%cd /kaggle/working/Arcadia
!git log --oneline -1

## 2. Install only what Kaggle is missing (torch is never reinstalled)

In [ ]:
import importlib.util, subprocess, sys

missing = [pkg for pkg in ("diffusers", "accelerate") if importlib.util.find_spec(pkg) is None]
print("installing:", missing or "nothing, all present")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## 3. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 4. Generate all 14 images

30 steps at 1344x768; roughly 30-60 s per image on a T4, plus a few minutes to download SDXL.

In [ ]:
!python -m render.generate_images --out /kaggle/working/images

## 5. List the results

In [ ]:
import json, glob
for path in sorted(glob.glob("/kaggle/working/images/*.json")):
    meta = json.load(open(path))
    print(f"{meta['id']:10s} seed {meta['seed']}  {meta['seconds']:5.1f}s  {meta['prompt'][:70]}")